In [1]:
# - --------------------------------------------------
# - -- Librerias
# - --------------------------------------------------

from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import os

In [2]:
# - --------------------------------------------------
# - -- Variables SparkCatalog/s3a
# - --------------------------------------------------

vType="hive"
vUri="thrift://localhost:9084"
vData="s3a://datalake"
vWarehouse="s3a://delta/warehouse"
vEndpoint="http://localhost:9000"
vAccessKey=os.getenv("AWS_ACCESS_KEY_ID")
vSecretKey=os.getenv("AWS_SECRET_ACCESS_KEY")

In [3]:
print("Servicios:")
print(f" Warehouse = {vWarehouse}")
print(f" EndPoint  = {vEndpoint}")
print(f" Uri       = {vUri}")

Servicios:
 Warehouse = s3a://delta/warehouse
 EndPoint  = http://localhost:9000
 Uri       = thrift://localhost:9084


In [4]:
# - --------------------------------------------------
# - -- Esquemas de base de datos para Lakehouse
# - --------------------------------------------------

vSchemaLand="landing" # - Capa Landing
vPathLand=f"{vWarehouse}/{vSchemaLand}/"
vSchemaBron="bronze"  # - Capa Bronze
vPathBron=f"{vWarehouse}/{vSchemaBron}/"
vSchemaSilv="silver"  # - Capa Silver
vPathSilv=f"{vWarehouse}/{vSchemaSilv}/"
vSchemaGold="gold"    # - Capa Gold
vPathGold=f"{vWarehouse}/{vSchemaGold}/"

In [5]:
print(f"Base de Datos Landing =  {vSchemaLand}")
print(f"Base de Datos Bronze  =  {vSchemaBron}")
print(f"Base de Datos Silver  =  {vSchemaSilv}")
print(f"Base de Datos Gold    =  {vSchemaGold}")

Base de Datos Landing =  landing
Base de Datos Bronze  =  bronze
Base de Datos Silver  =  silver
Base de Datos Gold    =  gold


In [6]:
# - --------------------------------------------------
# - -- Spark Session
# - --------------------------------------------------

def spark_session_hive(app_name):    
    builder = SparkSession.builder \
        .appName(app_name) \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .config("hive.metastore.uris", vUri) \
        .config("spark.hadoop.fs.s3a.endpoint", vEndpoint) \
        .config("spark.hadoop.fs.s3a.access.key", vAccessKey) \
        .config("spark.hadoop.fs.s3a.secret.key", vSecretKey) \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.sql.warehouse.dir", vWarehouse) \
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    
    return builder.enableHiveSupport().getOrCreate()    

In [7]:
# - --------------------------------------------------
# - -- Stop Spark Session
# - --------------------------------------------------

def stop_session(spark):
    if spark is not None:
        try:
            spark.stop()
            print("Sesión de Spark cerrada correctamente.")
        except Exception as e:
            print(f'{e}')        

In [8]:
# - --------------------------------------------------
# - -- Funcion(es) en Python
# - --------------------------------------------------

# - Nombre
def combinar_nombre_apellido(nombre, apellido):
    return f"{nombre} {apellido}"

# - Telefono
def formato_telefono(telefono):    
    tel = str(telefono)
    return '('+tel[0:3]+') '+tel[3:6]+'-'+tel[6:10]

# - Genero
def genero_completo(genero):
    if genero == "M":
        return "HOMBRE"
    if genero == "F":
        return "MUJER"
    else:
        return "DESCONOCIDO"

# - Grupo de edad
def grupo_edad(edad):
    if edad < 20:
        return "MENOR DE 20"
    if (edad >= 20) & (edad <= 29):
        return "DE 20 A 29"
    if (edad >= 30) & (edad <= 39):
        return "DE 30 A 39"
    if (edad >= 40) & (edad <= 49):
        return "DE 40 A 49"
    if (edad >= 50) & (edad <= 60):
        return "DE 50 A 60"
    else:
        return "MAYOR DE 60"

# - Merge 
def iceberg_upsert(df_source, target_table, join_columns, partition_column=None):
    spark = df_source.sparkSession
    view_name = f'view_{target_table.rsplit(".",1)[-1]}'    
    df_source.createOrReplaceTempView(view_name)
    on_condition = " AND ".join([f'target.{c} = source.{c}' for c in join_columns])
    if partition_column:
        on_condition += f' AND target.{partition_column} = source.{partition_column}'
    spark.sql(f"""
    MERGE INTO {target_table} AS target
    USING {view_name} AS source
    ON {on_condition}
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")

In [9]:
# - --------------------------------------------------
# - -- Registro de funcion(es)
# - --------------------------------------------------

# - Nombre
combinar_nombre_apellido_udf = udf(combinar_nombre_apellido, StringType())

# - Telefono
formato_telefono_udf = udf(formato_telefono, StringType())

# - Genero
genero_completo_udf = udf(genero_completo, StringType())

# - Grupo de edad
grupo_edad_udf = udf(grupo_edad, StringType())